In [1]:
import os
import json
import numpy as np
import pandas as pd
import torch

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
)
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

c:\Program Files\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W0420 11:46:05.698000 5604 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


In [2]:
MODEL_NAME = "bert-base-uncased"
DATASET_NAME = "nyu-mll/glue"
DATASET_CONFIG = "sst2"

OUT_DIR = "module6_sentiment_bert"
MAX_LEN = 128

# Keep it fast
TRAIN_SIZE = 2000
VAL_SIZE = 500
TEST_SIZE = 500

os.makedirs(OUT_DIR, exist_ok=True)

In [3]:
raw = load_dataset(DATASET_NAME, DATASET_CONFIG)

train_ds = raw["train"].shuffle(seed=42).select(range(min(TRAIN_SIZE, len(raw["train"]))))
val_ds = raw["validation"].shuffle(seed=42).select(range(min(VAL_SIZE, len(raw["validation"]))))

'(MaxRetryError('HTTPSConnectionPool(host=\'huggingface.co\', port=443): Max retries exceeded with url: /datasets/nyu-mll/glue/resolve/main/README.md (Caused by NameResolutionError("HTTPSConnection(host=\'huggingface.co\', port=443): Failed to resolve \'huggingface.co\' ([Errno 11001] getaddrinfo failed)"))'), '(Request ID: 88ee99d2-dbc0-468d-9ccc-4e64cf887772)')' thrown while requesting HEAD https://huggingface.co/datasets/nyu-mll/glue/resolve/main/README.md
Retrying in 1s [Retry 1/5].
'(MaxRetryError('HTTPSConnectionPool(host=\'huggingface.co\', port=443): Max retries exceeded with url: /datasets/nyu-mll/glue/resolve/main/README.md (Caused by NameResolutionError("HTTPSConnection(host=\'huggingface.co\', port=443): Failed to resolve \'huggingface.co\' ([Errno 11001] getaddrinfo failed)"))'), '(Request ID: 34669b53-a28f-4d47-a49f-a86ca82199be)')' thrown while requesting HEAD https://huggingface.co/datasets/nyu-mll/glue/resolve/main/README.md
Retrying in 2s [Retry 2/5].
'(MaxRetryError(

In [4]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

c:\Program Files\Python310\lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
'(MaxRetryError("HTTPSConnectionPool(host='huggingface.co', port=443): Max retries exceeded with url: /bert-base-uncased/resolve/main/tokenizer_config.json (Caused by SSLError(SSLCertVerificationError(1, '[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: self signed certificate in certificate chain (_ssl.c:1007)')))"), '(Request ID: ff04adad-0e5f-4405-8e51-9effed3deab1)')' thrown while requesting HEAD https://huggingface.co/bert-base-uncased/resolve/main/tokenizer_config.json
Retrying in 1s [Retry 1/5].
'(MaxRetryError("HTTPSConnectionPool(host='huggingface.co', port=443): Max retries exceeded with url: /bert-base-uncased/resolve/main/tokenizer_config.json (Caused by SSLError(SSLCertVerificati

In [5]:
def tokenize_batch(batch):
    return tokenizer(batch["sentence"], truncation=True, max_length=MAX_LEN)

train_tok = train_ds.map(tokenize_batch, batched=True)
val_tok = val_ds.map(tokenize_batch, batched=True)

# rename label -> labels for Trainer compatibility
train_tok = train_tok.rename_column("label", "labels")
val_tok = val_tok.rename_column("label", "labels")

In [6]:
cols_to_keep = {"input_ids", "attention_mask", "labels", "token_type_ids"}
train_cols = [c for c in train_tok.column_names if c not in cols_to_keep]
val_cols = [c for c in val_tok.column_names if c not in cols_to_keep]

train_tok = train_tok.remove_columns(train_cols)
val_tok = val_tok.remove_columns(val_cols)

train_tok.set_format("torch")
val_tok.set_format("torch")

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [7]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2
)

# SST-2 labels: 0 = negative, 1 = positive
model.config.id2label = {0: "negative", 1: "positive"}
model.config.label2id = {"negative": 0, "positive": 1}

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [8]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    acc = accuracy_score(labels, preds)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        preds,
        average="binary",
        zero_division=0
    )

    return {
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1,
    }

In [9]:
training_args = TrainingArguments(
    output_dir=OUT_DIR,
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=1,
    weight_decay=0.01,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    logging_steps=25,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    greater_is_better=True,
    report_to="none",
)

In [10]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tok,
    eval_dataset=val_tok,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()

 22%|██▏       | 27/125 [00:03<00:11,  8.60it/s]

{'loss': 0.6825, 'grad_norm': 9.308613777160645, 'learning_rate': 1.6000000000000003e-05, 'epoch': 0.2}


 42%|████▏     | 52/125 [00:06<00:07,  9.13it/s]

{'loss': 0.548, 'grad_norm': 7.029524326324463, 'learning_rate': 1.2e-05, 'epoch': 0.4}


 62%|██████▏   | 77/125 [00:09<00:05,  9.29it/s]

...

                                                 
100%|██████████| 125/125 [00:15<00:00,  8.79it/s]Checkpoint destination directory module6_sentiment_bert\checkpoint-125 already exists and is non-empty. Saving will proceed but saved results may be invalid.


{'eval_loss': 0.32360488176345825, 'eval_accuracy': 0.87, 'eval_precision': 0.8888888888888888, 'eval_recall': 0.8582375478927203, 'eval_f1': 0.8732943469785575, 'eval_runtime': 0.7597, 'eval_samples_per_second': 658.143, 'eval_steps_per_second': 21.061, 'epoch': 1.0}


100%|██████████| 125/125 [00:16<00:00,  7.40it/s]

{'train_runtime': 16.8811, 'train_samples_per_second': 118.475, 'train_steps_per_second': 7.405, 'train_loss': 0.4616425018310547, 'epoch': 1.0}


TrainOutput(global_step=125, training_loss=0.4616425018310547, metrics={'train_runtime': 16.8811, 'train_samples_per_second': 118.475, 'train_steps_per_second': 7.405, 'train_loss': 0.4616425018310547, 'epoch': 1.0})

In [11]:
metrics = trainer.evaluate()
print(metrics)

with open(os.path.join(OUT_DIR, "metrics.json"), "w", encoding="utf-8") as f:
    json.dump(metrics, f, indent=2)

100%|██████████| 16/16 [00:00<00:00, 23.14it/s]

{'eval_loss': 0.32360488176345825, 'eval_accuracy': 0.87, 'eval_precision': 0.8888888888888888, 'eval_recall': 0.8582375478927203, 'eval_f1': 0.8732943469785575, 'eval_runtime': 0.7439, 'eval_samples_per_second': 672.102, 'eval_steps_per_second': 21.507, 'epoch': 1.0}


In [12]:
def predict_sentiment(texts):
    if isinstance(texts, str):
        texts = [texts]

    enc = tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=MAX_LEN,
        return_tensors="pt"
    )

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    enc = {k: v.to(device) for k, v in enc.items()}

    model.eval()
    with torch.no_grad():
        logits = model(**enc).logits
        probs = torch.softmax(logits, dim=-1).cpu().numpy()
        preds = np.argmax(probs, axis=-1)

    outputs = []
    for text, pred, prob in zip(texts, preds, probs):
        outputs.append({
            "text": text,
            "prediction": model.config.id2label[int(pred)],
            "negative_prob": float(prob[0]),
            "positive_prob": float(prob[1]),
        })
    return outputs


In [13]:
samples = [
    "Shall daub her lips with her own children's blood",
    "And furious close of civil butchery",
    "As now we meet. You have deceived our trust",
    "me, so, 'tis a point of friendship"
]

In [14]:
results = predict_sentiment(samples)
for r in results:
    print(r)

pd.DataFrame(results).to_csv(os.path.join(OUT_DIR, "sample_predictions.csv"), index=False, encoding="utf-8")

{'text': "Shall daub her lips with her own children's blood", 'prediction': 'negative', 'negative_prob': 0.8428440690040588, 'positive_prob': 0.15715593099594116}
{'text': 'And furious close of civil butchery', 'prediction': 'negative', 'negative_prob': 0.8979839086532593, 'positive_prob': 0.1020161360502243}
{'text': 'As now we meet. You have deceived our trust', 'prediction': 'negative', 'negative_prob': 0.8562448024749756, 'positive_prob': 0.1437552273273468}
{'text': "me, so, 'tis a point of friendship", 'prediction': 'positive', 'negative_prob': 0.07876034826040268, 'positive_prob': 0.9212396144866943}
